In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as skl
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import metrics as ms
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,RobustScaler
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,roc_auc_score,confusion_matrix
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df=pd.read_pickle(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Datas\EDA_processed.pkl')

In [ ]:
df['debt_ratio__x'] = df['debt_ratio__x'].replace([np.inf, -np.inf], np.nan)

In [ ]:
X_train, X_test, y_train, y_test =train_test_split(
    df.drop(columns=['SK_ID_CURR','target']),
    df['target'],
    test_size=0.20,
    random_state=42,
    stratify=df['target'])

In [ ]:
id_col     = 'SK_ID_CURR'
target_col = 'target'


df['credit_segment'] = df['credit_segment'].astype(str)

df['DAYS_EMPLOYED_ANOM'] = df['DAYS_EMPLOYED_ANOM'].astype(float)

df['organization_type'] = df['organization_type'].fillna('Unknown')

# Grouping to categories for encoding

# Binary — 2 uniques, Y/N tipli
binary_cols = [
    'flag_own_car',      # Y / N
    'flag_own_realty',   # Y / N
]

# Ordinal
ordinal_cols  = ['name_education_type']
ordinal_order = [[
    'Lower secondary',
    'Secondary / secondary special',
    'Incomplete higher',
    'Higher education',
    'Academic degree'
]]

# OHE — (One Hot Encoding)
ohe_cols = [
    'name_contract_type',   # 2 unique
    'code_gender',          # 2 unique (F/M)
    'name_type_suite',      # 7 unique
    'name_income_type',     # 8 unique
    'name_family_status',   # 6 unique
    'name_housing_type',    # 6 unique
    'age_group',            # 4-5 unique
    'credit_segment'        # category → string
]

# TargetEncoder — many uniques
target_enc_cols = [
    'occupation_type',    # 18 unique
    'organization_type'   # 58 unique
]

# Numeric —  except SK_ID_CURR and target
numeric_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns.tolist()
    if c not in [id_col, target_col, 'DAYS_EMPLOYED_ANOM']
]
# DAYS_EMPLOYED_ANOM — binary numeric
binary_numeric_cols = ['DAYS_EMPLOYED_ANOM']


In [ ]:
te = TargetEncoder(cols=target_enc_cols, smoothing=20)
X_train_te = te.fit_transform(X_train, y_train)
X_test_te  = te.transform(X_test)

In [ ]:
binary_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OrdinalEncoder(
        handle_unknown= 'use_encoded_value',
        unknown_value=-1
    ))
])

ordinal_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OrdinalEncoder(
        categories=ordinal_order,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

ohe_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(
        drop='first',
        sparse_output=False,
        handle_unknown='ignore'
    ))
])

numeric_pipe = Pipeline(steps=[
                ('imputer',SimpleImputer(strategy='median')),
                ('scaler',RobustScaler())
])


binary_numeric_pipe = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent'))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', binary_pipe, binary_cols),
        ('ordinal', ordinal_pipe, ordinal_cols),
        ('ohe',ohe_pipe,ohe_cols),
        ('numeric',numeric_pipe,numeric_cols),
        ('binary_numeric',binary_numeric_pipe,binary_numeric_cols)
    ],remainder='drop'
)

In [ ]:
model_params = {
    # 'RandomForest': {
    #     'model': RandomForestClassifier(random_state=42, n_jobs=-1),
    #     'params': {
    #         'model__n_estimators': [100, 200, 300],
    #         'model__max_depth': [5, 10, 20],
    #         'model__min_samples_leaf': [1, 5, 10]
    #     }
    # },
    # 'XGBoost': {
    #     'model': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    #     'params': {
    #         'model__n_estimators': [100, 200],
    #         'model__learning_rate': [0.01, 0.05, 0.2],
    #         'model__max_depth': [3, 6, 9],
    #         'model__subsample': [0.6,0.8,1],
    #         'model__min_child_weight': [1,5,10]
    #     }
    # },

    'LightGBM':{
        'model':LGBMClassifier(random_state=42,
                               verbose=-1,
                               class_weight={0:1,1:10}),
        'params':{
            'model__n_estimators'    : [200, 500, 1000],
            'model__learning_rate'   : [0.01, 0.03, 0.05],
            'model__num_leaves'      : [15, 31, 63],
            'model__min_data_in_leaf': [20, 50, 100],
            'model__reg_alpha'       : [0, 0.1, 0.5],
            'model__reg_lambda'      : [0, 0.1, 0.5],
        }
    }
}

In [ ]:
performance_results = []
best_pipeline ={}
THRESHOLD = 0.35

for name,mp in model_params.items():
    print(f'Hal hazirda {name} ucun en yaxsi parameterler axtarilir...')

    full_pipeline = Pipeline(steps=[
        ('preprocessor',preprocessor),
        ('variance',VarianceThreshold(threshold=0.01)),
        ('model',mp['model'])
    ])

    rs =RandomizedSearchCV(
        estimator=full_pipeline,
        param_distributions=mp['params'],
        scoring='roc_auc',
        verbose=0,
        cv=3,
        n_iter=10,
        n_jobs=-1,
        random_state=42
    )

    rs.fit(X_train_te,y_train)


      
    best_pipe =rs.best_estimator_
    y_proba_m = best_pipe.predict_proba(X_test_te)[:,1]
    y_pred_m = (y_proba_m >= THRESHOLD).astype(int)

    test_auc = roc_auc_score(y_test,y_proba_m)

    row = {
            'Model'        : name,
            'Best_Params'  : rs.best_params_,
            'CV_AUC'       : round(rs.best_score_, 4),
            'Test_AUC'     : round(test_auc, 4),
            'Gini'         : round(2 * test_auc - 1, 4),
            'Recall'       : round(ms.recall_score(y_test, y_pred_m, pos_label=1), 4),
            'Precision'    : round(ms.precision_score(y_test, y_pred_m,
                                pos_label=1, zero_division=0), 4),
            'F1'           : round(ms.f1_score(y_test, y_pred_m,
                                pos_label=1, zero_division=0), 4),
            'Avg_Precision': round(ms.average_precision_score(y_test, y_proba_m), 4),
        }


    performance_results.append(row)
    best_pipeline[name] = best_pipe

    print(f"  CV AUC   : {row['CV_AUC']}")
    print(f"  Test AUC : {row['Test_AUC']}")
    print(f"  Gini     : {row['Gini']}")
    print(f"  Recall   : {row['Recall']}")



perf_df = pd.DataFrame(performance_results).sort_values(
    'Test_AUC', ascending=False
)

print(f"\n{'='*65}")
print("  FINAL NƏTICƏLƏR")
print(f"{'='*65}")
print(perf_df[['Model','CV_AUC','Test_AUC','Gini',
               'Recall','Precision','F1','Avg_Precision']].to_string(index=False))
print(f"\n🏆 Ən yaxşı model: {perf_df.iloc[0]['Model']} "
      f"(AUC={perf_df.iloc[0]['Test_AUC']})")



In [ ]:
y_proba_final = best_pipeline['LightGBM'].predict_proba(X_test_te)[:, 1]

print(f"{'Threshold':<10} | {'Recall':<10} | {'Precision':<10} | {'F1':<10}")
print("-" * 50)
for t in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
    yp = (y_proba_final >= t).astype(int)
    r  = ms.recall_score(y_test, yp, pos_label=1)
    p  = ms.precision_score(y_test, yp, pos_label=1, zero_division=0)
    f  = ms.f1_score(y_test, yp, pos_label=1, zero_division=0)
    print(f"{t:<10} | {r:<10.4f} | {p:<10.4f} | {f:<10.4f}")

In [ ]:


# ── Threshold-lar ─────────────────────────────────────────────
APPROVE_THRESHOLD  = 0.20   # PD < 0.20  → avtomatik təsdiq
DECLINE_THRESHOLD  = 0.50   # PD > 0.50  → avtomatik imtina

# 0.20 - 0.50 arası → kredit mütəxəssisi baxır

# ── 3-Tier klassifikasiya ─────────────────────────────────────
def classify_three_tier(proba):
    if proba < APPROVE_THRESHOLD:
        return 'APPROVE'
    elif proba < DECLINE_THRESHOLD:
        return 'REVIEW'
    else:
        return 'DECLINE'

y_tier = pd.Series(y_proba_final).apply(classify_three_tier)

# ── Tier DataFrame ────────────────────────────────────────────
tier_df = pd.DataFrame({
    'tier'          : y_tier.values,
    'proba'         : y_proba_final,
    'actual_target' : y_test.values,
    'amt_credit'    : df.loc[X_test.index, 'amt_credit'].values
})

# ── Tier analizi ──────────────────────────────────────────────
summary = tier_df.groupby('tier').agg(
    musteri_sayi  = ('proba',         'count'),
    default_sayi  = ('actual_target', 'sum'),
    default_rate  = ('actual_target', 'mean'),
    ort_kredit    = ('amt_credit',    'mean'),
    umumi_kredit  = ('amt_credit',    'sum')
).round(3)

summary['kredit_%'] = (
    summary['umumi_kredit'] /
    summary['umumi_kredit'].sum() * 100
).round(1)

print(f"{'='*70}")
print("  3-TİER KREDİT QƏRARI SİSTEMİ")
print(f"{'='*70}")
print(summary.to_string())

# ── Biznes cost function ──────────────────────────────────────
COST_FN = 50_000    # default qaçırmaq — kredit itirildi
COST_FP = 8_000     # yaxşı müştəriyə imtina — faiz itirildi
# men burda oz datasetime uygun olaraq deyerler vermek isteyirdim ,amt_credit mean taparaq yuxari kredit mebleglerine daha diqqetli yanasib derinden analiz elesin ,asagi qiymetli kreditlere ise o qeder ciddi yanasmayacaq, hem de bu gelecekde modelin monitoringini cetinlesdirecek hem de sabitliyini pozacaq

results_cost = []
for t in np.arange(0.10, 0.70, 0.01):
    yp          = (y_proba_final >= t).astype(int)
    fn          = ((yp == 0) & (y_test == 1)).sum()
    fp          = ((yp == 1) & (y_test == 0)).sum()
    total_cost  = fn * COST_FN + fp * COST_FP
    results_cost.append({
        'threshold' : round(t, 2),
        'fn'        : int(fn),
        'fp'        : int(fp),
        'total_cost': total_cost
    })

cost_df     = pd.DataFrame(results_cost)
optimal_idx = cost_df['total_cost'].idxmin()
optimal_t   = cost_df.loc[optimal_idx, 'threshold']
optimal_cost= cost_df.loc[optimal_idx, 'total_cost']

print(f"\n{'='*70}")
print("  OPTİMAL THRESHOLD — BİZNES COST FUNCTİON")
print(f"{'='*70}")
print(f"  Default qaçırmaq (FN) xərci : {COST_FN:,} AZN")
print(f"  Yanlış imtina    (FP) xərci : {COST_FP:,} AZN")
print(f"\n  Optimal threshold            : {optimal_t}")
yp_opt = (y_proba_final >= optimal_t).astype(int)
print(f"  FN (qaçırılan default)       : {cost_df.loc[optimal_idx,'fn']:,}")
print(f"  FP (yanlış imtina)           : {cost_df.loc[optimal_idx,'fp']:,}")
print(f"  Recall                       : {ms.recall_score(y_test, yp_opt, pos_label=1):.3f}")
print(f"  Precision                    : {ms.precision_score(y_test, yp_opt, pos_label=1, zero_division=0):.3f}")
print(f"  Ümumi biznes itkisi          : {optimal_cost:,.0f} AZN")

# ── Vizual ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sol — Cost function
axes[0].plot(cost_df['threshold'], cost_df['total_cost'],
             color='#e74c3c', linewidth=2)
axes[0].axvline(x=optimal_t, color='black', linestyle='--',
                label=f'Optimal = {optimal_t}')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Biznes itkisi (AZN)')
axes[0].set_title('Cost Function — Optimal Threshold', fontweight='bold')
axes[0].legend()
axes[0].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))

# Orta — 3 Tier müştəri sayı
tier_order  = ['APPROVE', 'REVIEW', 'DECLINE']
tier_colors = {'APPROVE': '#2ecc71', 'REVIEW': '#f39c12', 'DECLINE': '#e74c3c'}
tier_counts = y_tier.value_counts().reindex(tier_order)

bars = axes[1].bar(
    tier_counts.index,
    tier_counts.values,
    color=[tier_colors[t] for t in tier_counts.index],
    edgecolor='white', linewidth=1.5
)
axes[1].set_title('3-Tier Müştəri Bölgüsü', fontweight='bold')
axes[1].set_ylabel('Müştəri sayı')
for bar, val in zip(bars, tier_counts.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 100,
        f'{val:,}\n({val/len(y_tier)*100:.1f}%)',
        ha='center', fontsize=9, fontweight='bold'
    )

# Sağ — Default rate by tier
tier_dr = tier_df.groupby('tier')['actual_target'].mean().reindex(tier_order)
bars2   = axes[2].bar(
    tier_dr.index,
    tier_dr.values,
    color=[tier_colors[t] for t in tier_dr.index],
    edgecolor='white', linewidth=1.5
)
axes[2].axhline(y=y_test.mean(), color='black',
                linestyle='--', label=f'Ümumi: {y_test.mean():.2%}')
axes[2].set_title('Default Rate — Tier üzrə', fontweight='bold')
axes[2].set_ylabel('Default Rate')
axes[2].yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f'{x:.1%}'))
axes[2].legend()
for bar, val in zip(bars2, tier_dr.values):
    axes[2].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.002,
        f'{val:.1%}', ha='center', fontsize=10, fontweight='bold'
    )

plt.suptitle(
    f'3-Tier Biznes Qərar Sistemi | Optimal Threshold: {optimal_t}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

# ── Final DS qərarı ───────────────────────────────────────────
print(f"""
{'='*70}
  DS QƏRARI — REAL BANK MÜHİTİ
{'='*70}
  ✅ APPROVE  : PD < {APPROVE_THRESHOLD}  → avtomatik təsdiq
     Model əmindir, kredit mütəxəssisinin vaxtı sərf edilmir

  ⚠️  REVIEW   : {APPROVE_THRESHOLD} ≤ PD < {DECLINE_THRESHOLD} → kredit mütəxəssisi baxır
     Əlavə sənəd istənə bilər (gəlir sübut, girov)
     Final qərar insandır — model yalnız köməkçidir

  ❌ DECLINE  : PD ≥ {DECLINE_THRESHOLD}  → avtomatik imtina
     Model əmindir, kredit verilmir

  Optimal biznes threshold : {optimal_t}
  Gözlənilən biznes itkisi : {optimal_cost:,.0f} AZN (test datasında)
  
  Precision aşağı olmasının səbəbi:
  Dataset 92% non-default → bu credit risk-in real həyat problemidir
  Həll: 3-tier sistem + cost function optimizasiyası
{'='*70}
""")

In [ ]:
import shap
# ── Model və preprocessor-u pipeline-dan çıxart ──────────────
best_lgbm     = best_pipeline['LightGBM']
lgbm_model    = best_lgbm.named_steps['model']
preprocessor_ = best_lgbm.named_steps['preprocessor']

# ── Test datasını transform et ────────────────────────────────
X_test_processed = best_lgbm[:-1].transform(X_test_te)

# ── Feature adlarını al ───────────────────────────────────────
# OHE sütunlarının adları dəyişir — yeni adları alırıq
ohe_feature_names = (preprocessor_
                     .named_transformers_['ohe']
                     .named_steps['encoder']
                     .get_feature_names_out(ohe_cols)
                     .tolist())

all_feature_names = (
    binary_cols +
    ordinal_cols +
    ohe_feature_names +
    numeric_cols +
    binary_numeric_cols
)

# Variance threshold sonrası qalan feature-lar
variance_selector = best_lgbm.named_steps['variance']
feature_mask      = variance_selector.get_support()
final_feature_names = [f for f, m in zip(all_feature_names, feature_mask) if m]

print(f"Final feature sayı: {len(final_feature_names)}")
print(f"Processed shape   : {X_test_processed.shape}")

In [ ]:
sample_size = 500
np.random.seed(42)
sample_idx  = np.random.choice(X_test_processed.shape[0],
                                sample_size, replace=False)
X_sample    = X_test_processed[sample_idx]

print(f"SHAP sample shape: {X_sample.shape}")

explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_sample)

# LightGBM binary classification üçün:
# shap_values — list olarsa [0] non-default, [1] default
if isinstance(shap_values, list):
    sv = shap_values[1]   # default class üçün
else:
    sv = shap_values

print(f"SHAP values shape: {sv.shape}")
print("SHAP hesablandı ✓")

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv, X_sample,
    feature_names=final_feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Summary — Top 20 Feature', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv, X_sample,
    feature_names=final_feature_names,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance — Ortalama Təsir',
          fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
highest_risk_idx = np.argmax(y_proba_final[sample_idx])

print(f"Ən yüksək riskli müştəri:")
print(f"  Default ehtimalı : {y_proba_final[sample_idx[highest_risk_idx]]:.3f}")
print(f"  Həqiqi target    : {y_test.values[sample_idx[highest_risk_idx]]}")

shap.force_plot(
    explainer.expected_value[1] if isinstance(
        explainer.expected_value, list) else explainer.expected_value,
    sv[highest_risk_idx],
    X_sample[highest_risk_idx],
    feature_names=final_feature_names,
    matplotlib=True,
    show=False,
    figsize=(30, 3)
)
plt.title('Force Plot — Ən Riskli Müştəri', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
top_feature_idx = np.abs(sv).mean(0).argmax()
top_feature     = final_feature_names[top_feature_idx]

print(f"Ən vacib feature: {top_feature}")

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    top_feature_idx,
    sv, X_sample,
    feature_names=final_feature_names,
    show=False
)
plt.title(f'Dependency Plot — {top_feature}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#Müştəri üçün top risk faktorları
def explain_customer(customer_idx, proba, shap_vals, feature_names, n=5):
    factors = pd.DataFrame({
        'feature'  : feature_names,
        'shap_val' : shap_vals[customer_idx]
    })
    factors['abs']       = factors['shap_val'].abs()
    factors['direction'] = factors['shap_val'].apply(
        lambda x: '⬆️ riski artırır' if x > 0 else '⬇️ riski azaldır'
    )
    factors = factors.sort_values('abs', ascending=False).head(n)

    tier = ('✅ APPROVE' if proba < 0.20 else
            '⚠️  REVIEW'  if proba < 0.50 else
            '❌ DECLINE')

    print(f"\n{'='*55}")
    print(f"  MÜŞTƏRİ İZAHATI")
    print(f"{'='*55}")
    print(f"  Default ehtimalı : {proba:.3f}")
    print(f"  Qərar            : {tier}")
    print(f"  EL               : {proba * 0.45 * 300_000:,.0f} AZN")
    print(f"\n  Top {n} Risk Faktoru:")
    print(f"  {'-'*50}")
    for _, row in factors.iterrows():
        print(f"  {row['feature']:<30} {row['direction']}  "
              f"({row['shap_val']:+.4f})")
    print(f"{'='*55}")

# Ən riskli müştəri
explain_customer(
    highest_risk_idx,
    y_proba_final[sample_idx[highest_risk_idx]],
    sv,
    final_feature_names
)

# Ən az riskli müştəri
lowest_risk_idx = np.argmin(y_proba_final[sample_idx])
explain_customer(
    lowest_risk_idx,
    y_proba_final[sample_idx[lowest_risk_idx]],
    sv,
    final_feature_names
)

# ── SHAP explainer-i saxla ────────────────────────────────────
import joblib
joblib.dump(explainer, 'models/shap_explainer.pkl')
print("\nSHAP explainer saxlandı ✓")

In [ ]:
import os, joblib, json

# ── Qovluğu yarat ─────────────────────────────────────────────
os.makedirs('models', exist_ok=True)
print(f"Qovluq: {os.path.abspath('models')}")

# ── Saxla ─────────────────────────────────────────────────────
joblib.dump(te,                        'models/target_encoder.pkl')
joblib.dump(best_pipeline['LightGBM'], 'models/credit_pipeline.pkl')
joblib.dump(explainer,                 'models/shap_explainer.pkl')

config = {
    'model'             : 'LightGBM',
    'version'           : '1.0',
    'auc_roc'           : round(roc_auc_score(y_test, y_proba_final), 4),
    'gini'              : round(2 * roc_auc_score(y_test, y_proba_final) - 1, 4),
    'imbalance_strategy': 'class_weight_balanced',
    'lgd'               : 0.45,
    'thresholds'        : {
        'approve' : 0.20,
        'review'  : 0.50,
        'decline' : 0.50
    },
    'feature_names'  : final_feature_names,
    'target_enc_cols': target_enc_cols,
    'cost_fn'        : 50000,
    'cost_fp'        : 8000
}

with open('models/model_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("\nSaxlandı ✓")
print(f"  credit_pipeline.pkl  — {os.path.getsize('models/credit_pipeline.pkl')/1e6:.1f} MB")
print(f"  target_encoder.pkl   — {os.path.getsize('models/target_encoder.pkl')/1e6:.1f} MB")
print(f"  shap_explainer.pkl   — {os.path.getsize('models/shap_explainer.pkl')/1e6:.1f} MB")
print(f"  model_config.json")

In [ ]:
# # Bir yuxarı qovluğa çıx
# joblib.dump(explainer_new, '../models/shap_explainer.pkl')
# print("shap_explainer.pkl saxlandı ✓")

# with open('../models/model_config.json') as f:
#     config = json.load(f)

# config['feature_names'] = final_feature_names

# with open('../models/model_config.json', 'w') as f:
#     json.dump(config, f, indent=2, ensure_ascii=False)

# print("model_config.json yeniləndi ✓")
# print(f"Feature sayı config-də: {len(config['feature_names'])}")

In [ ]:
df_final =pd.read_pickle(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Datas\cleaned_data.pkl')

In [ ]:
df_final

In [ ]:
# df_final-də SK_ID_CURR var mı?
print('SK_ID_CURR' in df_final.columns)
print('sk_id_curr' in df_final.columns)
print(df_final.columns[:5].tolist())

In [ ]:
import sys
import joblib
import sklearn
print(f"Python: {sys.version}")
print(f"numpy: {np.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"lightgbm: {lgb.__version__}")
print(f"joblib: {joblib.__version__}")

In [ ]:
# Əvvəlcə uyğun versiyaları yüklə
import subprocess
subprocess.run(['pip', 'install', 
                'numpy==1.26.4',
                'category-encoders==2.6.3',
                'scikit-learn==1.4.2',
                'lightgbm==4.3.0',
                '--quiet'])

In [ ]:
import joblib
import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Mövcud modelləri yüklə
te_old       = joblib.load(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Notebooks\models\target_encoder.pkl')
pipeline_old = joblib.load(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Notebooks\models\credit_pipeline.pkl')
explainer_old = joblib.load(r'C:\Users\Lenovo\Desktop\Home-Credit-Risk-Management\Notebooks\models\shap_explainer.pkl')

# Eyni obyektləri yenidən saxla — yeni versiya ilə
joblib.dump(te_old,        '../models/target_encoder.pkl')
joblib.dump(pipeline_old,  '../models/credit_pipeline.pkl')
joblib.dump(explainer_old, '../models/shap_explainer.pkl')

print(f"numpy versiyası: {np.__version__}")
print("Modellər yenidən saxlandı ✓")

In [ ]:
import numpy as np
import sklearn
import lightgbm as lgb
import joblib

print(f"numpy  : {np.__version__}")
print(f"sklearn: {sklearn.__version__}")
print(f"lgb    : {lgb.__version__}")
print(f"joblib : {joblib.__version__}")

In [ ]:
import joblib
import numpy as np

print(f"numpy: {np.__version__}")

# Modelləri yüklə — xəta verməməlidir
te       = joblib.load('../models/target_encoder.pkl')
pipeline = joblib.load('../models/credit_pipeline.pkl')

print("target_encoder.pkl ✓")
print("credit_pipeline.pkl ✓")
print("Hər şey yaxşıdır!")

In [ ]:
engine = sa.create_engine(
    DB_URI,
    pool_pre_ping=True,
    connect_args={
        "sslmode": "require",
        "client_encoding": "utf8"
    }
)